# 01 - Solver validation

The five sanity checks of §3.7, which are §11.2 steps 1-5, plus the two things that have
to hold before any of them mean anything: the configuration is internally consistent, and
the velocity-to-displacement deconvolution is conditioned across the whole operating band.

§11.3 lists the solver sanity checks under *never cut, regardless of time*. The reason is
not diligence for its own sake. A surrogate trained against an unvalidated solver is a
surrogate for the wrong operator, and every metric downstream -- relative L2, phase error,
inversion success rate -- will look fine while being an accurate report on the wrong
physics. Nothing later in the pipeline can detect that. This notebook is the only place
that can.

**Runtime.** Checks 1-3 are seconds. Checks 4 and 5 refine the grid and sweep a radius, so
they want a GPU: budget 20-40 minutes on an A100. `run_all` skips them on CPU, and this
notebook says so loudly rather than quietly reporting three passes out of five.

**Platform.** No Modal API calls appear in this notebook or any of the other five;
`bootstrap.setup()` works out where it is running. The headless equivalent of this
notebook is

```
modal run modal_app.py::solver_checks
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

## The configuration is the single source of truth

`config.self_check()` prints the grid, CFL, band, material, dispersion, parameter, ring
and inversion tables and then *asserts* every claim in them. It is not a report; it is a
refusal to continue if the numbers in §2-§8 and the numbers in the code have drifted apart.

Two values in the output are deliberate deviations from the document, and both are worth
recognising when they scroll past:

- The CFL number is **derived** from the 4th-order stencil's stability limit
  (`CFL_LIMIT_4TH = 0.6061`) times the stated 0.9 safety factor, not the document's quoted
  `0.6 dx/c`. That moves `NT` from 1280 to 1408 for the same 24 T_p window. Honouring the
  safety factor and honouring the quoted number are not the same thing, and the safety
  factor is the one that keeps the run stable.
- Everything is sized on the **shear** wavelength at the worst Poisson ratio (nu = 0.37),
  never on lambda_p. lambda_s is the shorter wave, so it sets points-per-wavelength,
  dispersion and the absorber thickness.

In [ ]:
cfg.self_check()

In [ ]:
from src.models.fno2d import band_in_modes

k_needed = band_in_modes()
print(f"parameter budget (primary variant)")
print(f"  config.total_params(d_v={cfg.D_V}, kmax={cfg.KMAX}, "
      f"n_blocks={cfg.N_BLOCKS}) = {cfg.total_params():,}")
print(f"\nmode truncation")
print(f"  band top f = {max(cfg.FREQS)/cfg.FC:.3f} f_c at nu = {min(cfg.NU_LIST)} needs "
      f"mode index {k_needed:.1f}")
print(f"  KMAX = {cfg.KMAX}  ->  {cfg.KMAX/k_needed:.2f}x headroom for near-field content")
print(f"  Nyquist on the {cfg.N_NET}^2 grid is {cfg.K_NYQUIST}, so the retained band is "
      f"strictly inside the resolved band")

## The deconvolution has to be conditioned, or steps 1-5 are measuring noise

The solver is a velocity-stress FDTD and runs a DFT inside the time loop, so what comes
out is a *velocity* phasor. Everything downstream -- the network's targets, the incident
cache, the inversion's data -- is a *displacement* phasor. The bridge is

    u_hat(omega) = v_hat(omega) / (i omega s_hat(omega))

and dividing by `s_hat` is a deconvolution. The 5-cycle Hann-windowed tone burst has
spectral nulls at 0.6 f_c and 1.4 f_c; the band runs 0.66 to 1.34 f_c, which sits *inside*
those nulls but not far inside. `conditioning_report` measures how much each line is
amplified relative to the strongest one, and `assert_conditioned` refuses anything worse
than `MAX_DECONV_AMPLIFICATION = 25`.

This is one of the three named failure modes of the time-harmonic reduction (the others
being quadrature mismatch, handled by the midpoint rule at t = (n + 1/2) dt, and
wrap-around, which check 3 and the tail-energy diagnostic in notebook 02 cover). It is
checked first because a band-edge line amplified 200x would turn every subsequent number
in this notebook into a report on round-off.

In [ ]:
from src.solver import harmonic as H

rep = H.conditioning_report()
f = _np(rep["freqs"])
s = _np(rep["s_hat_abs"])
a = _np(rep["amplification"])
worst_i = int(rep["worst_index"])

print(f"{'m':>3} {'f/f_c':>8} {'|s_hat|':>12} {'amplification':>14}")
for m in range(len(f)):
    flag = "   <-- worst" if m == worst_i else ""
    print(f"{m:>3} {f[m]:8.4f} {s[m]:12.4e} {a[m]:14.3f}{flag}")

print(f"\nworst amplification {float(rep['worst']):.2f} at m = {worst_i} "
      f"(f = {f[worst_i]:.3f} f_c), limit {H.MAX_DECONV_AMPLIFICATION}")
H.assert_conditioned()
print("PASS  the deconvolution is conditioned across the whole band")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.1))

ff = np.linspace(0.3, 1.7, 601)
om = torch.tensor(2.0 * np.pi * ff, dtype=torch.float64)
sh = H.source_hat(om).abs().numpy()
ax[0].semilogy(ff, sh / sh.max(), lw=1.0, color="0.35",
               label="|s_hat| (continuous)")
ax[0].semilogy(f, s / sh.max(), "o", ms=4, color="C0", label="the 20 band lines")
for x in (0.6, 1.4):
    ax[0].axvline(x, ls=":", c="C3", lw=1.0)
ax[0].axvspan(min(f), max(f), color="C0", alpha=0.08)
ax[0].set(xlabel="f / f_c", ylabel="|s_hat| (normalised)", ylim=(1e-4, 2.0),
          title="tone-burst spectrum\n(Hann nulls at 0.6 and 1.4 f_c, dotted)")
ax[0].legend(fontsize=7.5, loc="lower center")

ax[1].plot(f, a, "o-", ms=4)
ax[1].axhline(H.MAX_DECONV_AMPLIFICATION, ls="--", c="C3", lw=1.0,
              label=f"limit = {H.MAX_DECONV_AMPLIFICATION:g}")
ax[1].set(xlabel="f / f_c", ylabel="1 / |s_hat| relative to the strongest line",
          title="deconvolution amplification per line")
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "01_deconvolution_conditioning.png")
plt.show()

## §11.2 steps 1-5

| # | check | gate | what a failure would mean |
|---|-------|------|---------------------------|
| 1 | energy drift with the absorber removed | `< 0.5%` over the run | the update is not conservative: wrong Lame coefficients, a stencil bug, or dt above the stability limit |
| 2 | P and S arrival times against analytic | within one time step | the wave speeds are wrong, or the source is not where the acquisition table says it is |
| 3 | residual energy after the wave has left | `< 1e-4` of peak | the PML is reflecting; every A-scan carries a ghost of itself |
| 4 | Rayleigh scaling of scattered energy with radius | slope in `(3.3, 4.7)` **and** visible mode conversion | small voids are not being resolved -- the label floor is above the smallest defect in the dataset |
| 5 | grid convergence, `256^2` vs `512^2` | `< 2%` rel-L2 | the training labels are discretisation error, not physics |

Check 4's gate is a conjunction, and deliberately so. The theoretical slope is 4 (energy
goes as R^4 in the 2D Rayleigh limit, amplitude as R^2), the window brackets it, and a run
that lands the slope but produces no mode conversion has almost certainly done so by
scattering off a numerical artefact rather than off the void.

In [ ]:
from src.solver import validate as V

include_slow = DEV.startswith("cuda")
if not include_slow:
    print("NO GPU DETECTED.\n"
          "Checks 4 and 5 refine the grid and will be skipped, so this run reports\n"
          "3 of 5.  Both are on the never-cut list of §11.3 -- do not quote any\n"
          "result from a CPU-only run of this notebook.\n")

t0 = time.perf_counter()
results = V.run_all(device=DEV, include_slow=include_slow)
wall = time.perf_counter() - t0

print(f"\n{len(results)} checks in {wall/60:.1f} min on {DEV}\n")
for r in results:
    print(r)

by = {r.name.split(".")[0]: r for r in results}
n_pass = sum(1 for r in results if r.passed)
print(f"\n{n_pass} / {len(results)} passed")

### Check 1 -- energy conservation

Run on a domain of doubled side (L = 16) with the absorber removed, so nothing is
deliberately absorbing and any drift is the update's own. The gate is on the *drift*
between the start and the end of the window, not on the oscillation: a staggered
velocity-stress scheme stores kinetic and strain energy half a step apart, so the discrete
total oscillates at the sampling frequency by a bounded amount that does not accumulate.
Gating the oscillation would fail a correct solver.

In [ ]:
r = by.get("1")
if r is None:
    print("check 1 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    k0 = int(en.argmax())

    fig, ax = plt.subplots(1, 2, figsize=(9.0, 3.0))
    ax[0].plot(t, en_n, lw=0.9)
    ax[0].axvline(t[k0], ls=":", c="0.5", lw=1.0)
    ax[0].set(xlabel="t / T_p", ylabel="E / E_max",
              title=f"total energy, L = {r.extras['l_domain']:g}, no absorber")

    tail = slice(k0, None)
    ax[1].plot(t[tail], en_n[tail] / en_n[tail][0] - 1.0, lw=0.9)
    ax[1].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.1e}")
    ax[1].axhline(-r.gate, ls="--", c="C3", lw=1.0)
    ax[1].set(xlabel="t / T_p", ylabel="E(t)/E(peak) - 1",
              title=f"drift {r.value:.2e}   oscillation "
                    f"{float(r.extras['oscillation']):.2e}")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check1_energy.png")
    plt.show()

### Check 2 -- P and S arrival times

A 2-cycle burst (short enough that the P and S packets separate before either reaches the
far receivers) with arrivals picked from the analytic envelope peak, corrected for the
group delay of the burst itself, `N_c / (2 f_c)`. Picking the first threshold crossing
instead would measure the burst's rise time rather than the wave speed, and would drift
with amplitude.

Receivers marked unusable are ones where the P and S packets overlap -- too close to the
source for a 2-cycle burst to separate them. They are excluded rather than fudged.

In [ ]:
r = by.get("2")
if r is None:
    print("check 2 not in this run")
else:
    d = _np(r.extras["distances"])            # all 32 receivers
    ep = _np(r.extras["errs_p"])              # only the ones actually picked
    es = _np(r.extras["errs_s"])
    use = _np(r.extras["usable"]).astype(bool)
    asc = _np(r.extras["ascans"])

    # errs_p/errs_s are appended in receiver order over the usable receivers only,
    # and a receiver whose window runs off the end of the record is skipped as
    # well, so they are aligned with d[use] from the left but can be shorter.
    d_use = d[use]
    n = min(len(ep), len(es), len(d_use))
    if n < len(d_use):
        print(f"{len(d_use) - n} usable receiver(s) had no clean window "
              f"(the record ends before the predicted arrival); not plotted")

    fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.2))
    ax[0].plot(d_use[:n], ep[:n], "o", ms=4, label="P")
    ax[0].plot(d_use[:n], es[:n], "s", ms=4, label="S")
    if (~use).any():
        # These have no error to plot: the two packets overlap, so there is no
        # peak to pick.  Shade the range rather than invent a value for it.
        ax[0].axvspan(0.0, float(d[~use].max()), color="0.88", zorder=0,
                      label=f"excluded, P/S overlap ({int((~use).sum())} recv)")
    ax[0].axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:g} step")
    ax[0].set(xlabel="source-receiver distance / lambda_p",
              ylabel="|arrival error| (time steps)",
              xlim=(0.0, 1.02 * float(d.max())), ylim=(0.0, None),
              title=f"worst |error| = {r.value:.3f} steps")
    ax[0].legend(fontsize=7.5)

    # a few traces with their envelopes, to show the picking is on a clean packet
    trace = asc[0] if asc.ndim == 4 else asc
    pick = np.argsort(d)[::-1][:4]
    t_ax = np.arange(trace.shape[-1]) * cfg.DT
    for k, i in enumerate(pick):
        x = trace[i, 0]
        env = _np(H.envelope(torch.from_numpy(np.ascontiguousarray(x))))
        off = 1.15 * k
        norm = max(abs(x).max(), 1e-30)
        ax[1].plot(t_ax, x / norm + off, lw=0.7, c=f"C{k}")
        ax[1].plot(t_ax, env / norm + off, lw=1.0, c="0.25", alpha=0.8)
    ax[1].set(xlabel="t / T_p", ylabel="receiver (offset)", yticks=[],
              title="x-velocity and analytic envelope\n(4 most distant receivers)")
    fig.tight_layout()
    savefig(fig, "01_check2_arrivals.png")
    plt.show()

### Check 3 -- the absorber

Residual energy in the interior long after the wave has reached the boundary, as a
fraction of the peak. The graded polynomial PML is only reflectionless for waves that are
already propagating freely when they enter it, which is why the dataset's rejection
sampler keeps every void boundary `BOUNDARY_KEEPOUT_LS = 1.5` shear wavelengths away from
the walls -- a void inside the absorber would scatter into a medium that is not the medium
the absorber was designed for.

This number is also the wrap-around margin for the running DFT: the phasors are computed
over a finite window, so anything still ringing at the end of it aliases back onto the
band. Notebook 02 measures the same quantity on the real A-scans.

In [ ]:
r = by.get("3")
if r is None:
    print("check 3 not in this run")
else:
    t = _np(r.extras["t"])
    en = _np(r.extras["energy"])
    en_n = en / max(en.max(), 1e-300)
    frac = float(r.extras["tail_fraction"])

    fig, ax = plt.subplots(figsize=(5.6, 3.1))
    ax.semilogy(t, np.maximum(en_n, 1e-16), lw=0.9)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0e}")
    ax.axvspan(t[int(0.9 * len(t))], t[-1], color="C3", alpha=0.08,
               label="tail window")
    ax.set(xlabel="t / T_p", ylabel="E(t) / E_peak",
           title=f"absorber residual {r.value:.2e}   tail fraction {frac:.2e}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check3_absorber.png")
    plt.show()

### Check 4 -- the Rayleigh limit and mode conversion

Scattered energy against void radius on log-log axes. In the 2D Rayleigh regime
(`kR << 1`) the scattered amplitude goes as R^2 and the energy as R^4, so the slope of
`log E` against `log R` should be 4. The gate window `(3.3, 4.7)` is wide because the
largest radii in the sweep are leaving the Rayleigh regime and the smallest are approaching
the label floor from check 5. `extras["local_slope"]` is the slope of the smallest pair
alone; the middle panel below recomputes every consecutive pair from the energies, which is
what actually shows where the clean scaling lives.

The second condition is that mode conversion is visible: a P wave hitting a traction-free
curved boundary must produce an S wave. If it does not, the boundary is not being resolved
as a boundary, and the "scattered field" being measured is a staircase artefact that
happens to scale plausibly. The reported ratio is scattered energy in the S arrival window
over that in the P window, for the largest void in the sweep -- one number, not a sweep.

This check is what sets the **label floor**: the smallest radius whose scattered field is
above the discretisation error is the smallest defect the dataset can honestly contain.
`R_MIN_LS = 0.4` shear wavelengths sits above it.

In [ ]:
r = by.get("4")
if r is None:
    print("check 4 not in this run (needs a GPU)")
else:
    en = _np(r.extras["energy"])
    kR = _np(r.extras["kR"])
    slope = float(r.extras["slope"])
    # `radii` is in absolute (non-dimensional length) units, but the natural axis
    # here is radius in shear wavelengths -- and kR = 2 pi R / lambda_s gives it
    # exactly, without having to assume which nu the run used.
    rad = kR / (2.0 * np.pi)
    # These two are single numbers, not one per radius: `local_slope` is the slope
    # of the smallest pair, `mode_conversion` is one ratio over the largest void.
    loc_reported = float(r.extras["local_slope"])
    conv = float(r.extras["mode_conversion"])

    # Every consecutive pair, so the panel shows where the clean R^4 scaling lives
    # rather than only reporting the one pair that validate.py happens to return.
    lp = np.diff(np.log(en)) / np.diff(np.log(rad))
    mid = np.sqrt(rad[1:] * rad[:-1])                 # geometric midpoints

    fig, ax = plt.subplots(1, 3, figsize=(11.0, 3.1))

    ax[0].loglog(rad, en, "o-", ms=4)
    ref = en[0] * (rad / rad[0]) ** 4.0
    ax[0].loglog(rad, ref, ls="--", c="0.45", lw=1.0, label="slope 4 (Rayleigh)")
    ax[0].set(xlabel="R / lambda_s", ylabel="scattered energy",
              title=f"fitted slope {slope:.3f}\ngate window (3.3, 4.7)")
    ax[0].legend(fontsize=8)

    ax[1].semilogx(mid, lp, "o-", ms=4, label="consecutive pairs")
    ax[1].axhspan(3.3, 4.7, color="C2", alpha=0.12, label="gate window")
    ax[1].axhline(4.0, ls=":", c="0.4", lw=1.0)
    ax[1].plot([mid[0]], [loc_reported], "x", c="C3", ms=9, mew=1.6,
               label=f"reported {loc_reported:.2f}")
    ax[1].set(xlabel="R / lambda_s (pair midpoint)", ylabel="d log E / d log R",
              title="local slope, pair by pair")
    ax[1].legend(fontsize=7.5)

    # One number, so one bar.  A line plot of a scalar against the four kR values
    # would suggest a sweep that was never run.
    ax[2].bar([0], [conv], width=0.55, color="C0", alpha=0.85)
    ax[2].axhline(0.01, ls="--", c="C3", lw=1.0, label="gate: > 1%")
    ax[2].set(xticks=[0], xticklabels=[f"R = {rad[-1]:.3f} lambda_s"],
              ylabel="S-window energy / P-window energy",
              yscale="log", xlim=(-0.6, 0.6),
              title=f"mode conversion {conv:.1%}\nkR in "
                    f"[{kR.min():.2f}, {kR.max():.2f}]")
    ax[2].legend(fontsize=8)

    fig.tight_layout()
    savefig(fig, "01_check4_rayleigh.png")
    plt.show()
    print(f"grid {r.extras['grid']}  dx {float(r.extras['dx']):.5f}  "
          f"nt {int(r.extras['nt'])}")

### Check 5 -- grid convergence

The same scattering problem at `256^2` and at `512^2`, compared per frequency on the
network grid. This is the number that says the training labels are physics rather than
discretisation error, and it is reported per frequency because the error is not uniform
across the band: the top of the band has the fewest points per wavelength and converges
last. If any single line is above the gate, that line's labels are not trustworthy even if
the band average passes.

In [ ]:
r = by.get("5")
if r is None:
    print("check 5 not in this run (needs a GPU)")
else:
    pf = _np(r.extras["per_frequency"]).reshape(-1)
    fr = np.asarray(cfg.FREQS[:len(pf)])

    fig, ax = plt.subplots(figsize=(6.4, 3.1))
    ax.bar(fr, pf, width=0.9 * cfg.DF, color="C0", alpha=0.85)
    ax.axhline(r.gate, ls="--", c="C3", lw=1.0, label=f"gate {r.gate:.0%}")
    ax.set(xlabel="f / f_c", ylabel="rel-L2, 256^2 vs 512^2",
           title=f"grid convergence at R = {float(r.extras['radius']):.3f}, "
                 f"refine {int(r.extras['refine'])}x\nworst line {pf.max():.4f}, "
                 f"reported {r.value:.4f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "01_check5_convergence.png")
    plt.show()

## Gate summary

Written to `E.results` so notebook 06's thesis table can quote it without re-running the
slow checks. A `False` here is a stop, not a note: nothing downstream is meaningful until
it is a `True`.

In [ ]:
rows = [(r.name, f"{r.value:.4e}", f"{r.gate:.2e}", r.units or "-",
         "PASS" if r.passed else "FAIL") for r in results]
table(rows, ["check", "value", "gate", "units", ""])

record = {
    "device": DEV,
    "gpu": E.gpu_name,
    "include_slow": include_slow,
    "wall_minutes": wall / 60.0,
    "deconvolution": {"worst_amplification": float(rep["worst"]),
                      "worst_index": worst_i,
                      "limit": H.MAX_DECONV_AMPLIFICATION},
    "checks": {r.name: dict(passed=bool(r.passed), value=float(r.value),
                            gate=float(r.gate), units=r.units, detail=r.detail)
               for r in results},
    "n_pass": n_pass,
    "n_total": len(results),
}
dump(record, "01_solver_validation.json")

if not include_slow:
    print("\nINCOMPLETE: checks 4 and 5 were skipped.  Re-run on a GPU.")
elif n_pass == len(results):
    print("\nAll five solver checks pass.  The dataset in notebook 02 is worth "
          "generating.")
else:
    print("\nSTOP.  Fix the solver before generating a dataset -- a surrogate "
          "trained\non these labels would be an accurate model of the wrong "
          "operator.")

## If a gate fails

- **1 (energy).** Check `dt` against `CFL_LIMIT_4TH * CFL_SAFETY * dx / c_p` first; a
  marginally unstable run drifts slowly rather than exploding. Then check the Lame
  coefficients: `lame_from_nu` assumes `rho = c_p = 1`.
- **2 (arrivals).** Almost always the source position. `source_position` documents a
  deliberate `dx_fine/2` offset between the network-grid index and the physical
  coordinate; using the wrong one shifts every arrival by half a fine cell.
- **3 (absorber).** Increase `N_PML_FINE`, or reduce `pml_d0`. A residual just above the
  gate is usually the profile's grading exponent, not its thickness.
- **4 (Rayleigh).** If the slope is right but conversion is absent, the interface width
  `EPS_INTERFACE_CELLS` is too large relative to the radius being tested; if both fail at
  small R only, that is the label floor and `R_MIN_LS` needs raising.
- **5 (convergence).** Failing only at the top of the band means the network grid is
  under-resolved there; that is an argument for trimming `M_FREQ`, which notebook 02's
  size projection also suggests as its first lever.